# Lesson 02 - Exploring Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** เป็นกรอบงานแบบรวมศูนย์สำหรับการสร้างเอเจนต์ AI มันมีสถาปัตยกรรมที่สะอาดและประกอบได้ด้วยบล็อกแกนหลักสี่อย่าง:

- **Client** – เชื่อมต่อกับจุดสิ้นสุดของโมเดล AI และจัดการการสื่อสาร
- **Agent** – ครอบคลุม client ด้วยคำแนะนำและคำจำกัดความเครื่องมือ
- **Tools** – ขยายความสามารถของเอเจนต์ด้วยฟังก์ชันที่กำหนดเองที่โมเดลสามารถเรียกใช้ได้
- **Session** – บำรุงประวัติการสนทนาเพื่อการโต้ตอบหลายรอบ

ในบทเรียนนี้ เราจะสร้าง **เอเจนต์จองท่องเที่ยว** ที่ตรวจสอบความพร้อมใช้งานของปลายทางโดยใช้แนวคิดเหล่านี้


## การตั้งค่า


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## การทำความเข้าใจสถาปัตยกรรมของ Agent Framework

Microsoft Agent Framework ใช้สถาปัตยกรรมแบบเลเยอร์:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` เชื่อมต่อกับ Azure OpenAI deployment ดูแลการพิสูจน์ตัวตน การจัดรูปแบบคำขอ และการแยกวิเคราะห์คำตอบ
2. **Agent** – สร้างจาก client ผ่าน `provider.create_agent()` ตัว agent ผสมผสานการเข้าถึงโมเดลกับคำสั่ง (system prompt) และเครื่องมือ
3. **Tools** – ฟังก์ชัน Python ที่ตกแต่งด้วย `@tool` ซึ่ง agent สามารถเรียกใช้เพื่อทำงานหรือดึงข้อมูล
4. **Session** – วัตถุ `AgentSession` (สร้างผ่าน `agent.create_session()`) ที่เก็บประวัติการสนทนา ทำให้สามารถโต้ตอบหลายรอบโดย agent จำบริบทก่อนหน้าได้

เรามาสร้างแต่ละเลเยอร์ทีละขั้นตอนกันเถอะ


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การเพิ่มเครื่องมือด้วยตัวตกแต่ง @tool

เครื่องมือช่วยให้ตัวแทนดำเนินการได้มากกว่าการสร้างข้อความ ตัวตกแต่ง `@tool` จะแปลงฟังก์ชัน Python ธรรมดาให้กลายเป็นสิ่งที่ตัวแทนสามารถเรียกใช้ได้

ประเด็นสำคัญ:
- ใช้ `Annotated[type, "description"]` เพื่อให้โมเดลเข้าใจพารามิเตอร์แต่ละตัว
- docstring จะกลายเป็นคำอธิบายเครื่องมือที่โมเดลเห็น
- `approval_mode="never_require"` หมายความว่าเครื่องมือจะทำงานโดยอัตโนมัติโดยไม่ต้องรอการยืนยันจากผู้ใช้


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## การสร้าง Agent พร้อมเครื่องมือ

ตอนนี้เรารวมไคลเอนต์ คำแนะนำ และเครื่องมือเข้าด้วยกันเป็น agent โดย `instructions` ทำหน้าที่เป็น system prompt — ซึ่งกำหนดบุคลิกและพฤติกรรมของ agent


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## การสนทนาหลายตาระหว่างเซสชัน

`AgentSession` (สร้างโดย `agent.create_session()`) จะติดตามข้อความทั้งหมดในบทสนทนา โดยการส่งเซสชันเดียวกันไปยังแต่ละการเรียก `agent.run()` ตัวแทนจะสามารถเข้าถึงประวัติการสนทนาทั้งหมดและสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าได้

เราใส่ `tools=[check_destination_availability]` เพื่อให้ตัวแทนสามารถเรียกเครื่องมือตรวจสอบความพร้อมใช้งานของเราระหว่างแต่ละตาได้


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## สรุป

ในบทเรียนนี้คุณได้สำรวจเสาหลักสี่ประการของ Microsoft Agent Framework:

| แนวคิด | สิ่งที่คุณได้เรียนรู้ |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` เชื่อมต่อกับ Azure OpenAI ด้วยการตรวจสอบสิทธิ์แบบใช้ข้อมูลประจำตัว |
| **Agent** | `provider.create_agent()` รวมการเชื่อมต่อโมเดลกับคำสั่งและชื่อ |
| **Tools** | ตกแต่ง `@tool` เผยฟังก์ชัน Python สำหรับให้เอเย่นต์เรียกใช้ |
| **Session** | `agent.create_session()` รักษาประวัติการสนทนาข้ามหลายรอบ |

บล็อกส่วนประกอบเหล่านี้รวมกันเพื่อสร้างเอเย่นต์ที่สามารถสนทนาแบบธรรมชาติ เรียกฟังก์ชันภายนอก และรักษาบริบท — ซึ่งเป็นรากฐานสำหรับรูปแบบเอเย่นต์ขั้นสูงในบทเรียนถัดไป.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ข้อความแปลมีความถูกต้อง แต่โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาต้นฉบับถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้การแปลโดยผู้เชี่ยวชาญมนุษย์ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
